###Run Shared Libraries

In [0]:
%run ../Misc/sharedlibraries


In [0]:
UpdatedDateTime = datetime.datetime.now()
Entity = "dimdate"

###Read Bronze tables

In [0]:
fiscalperioddf = spark.table("bronze.fiscalperiod")

In [0]:
display(fiscalperioddf)

In [0]:
fiscalperioddf.printSchema()

####Date Dimension from Python

In [0]:
start_date = datetime.date(2018,1,1)
end_date = start_date + dateutil.relativedelta.relativedelta(years=8,month=12,day=31)

start_date = datetime.datetime.strptime(
    f"{start_date}", "%Y-%m-%d"
)

end_date = datetime.datetime.strptime(
    f"{end_date}", "%Y-%m-%d"
)

print(start_date)
print(end_date)

In [0]:
datepddf = pd.date_range(start_date,end_date, freq='D').to_frame(name='Date')
print(datepddf)

In [0]:
datepddf = pd.date_range(start_date,end_date, freq='D').to_frame(name='Date')
datedf = spark.createDataFrame(datepddf)

display(datedf)

In [0]:
joindf = (
    datedf.join(
        fiscalperioddf.filter(fiscalperioddf.RecordId.isNotNull()),
        (datedf.Date >= fiscalperioddf.FiscalStartDate)
        & (datedf.Date <= fiscalperioddf.FiscalEndDate),
        "left"
    )
)

display(joindf)

###Build Date Dimension

In [0]:
datedimdf = joindf.select(
    F.col("Date"),
    F.date_format(F.col("Date"), "yyyyMMdd").cast("int").alias("DateId"), #PK
    F.year(F.col("Date")).alias("Year"),
    F.month(F.col("Date")).alias("Month"),
    F.date_format(F.col("Date"), "MMM").cast("string").alias("MonthName"),
    F.dayofmonth(F.col("Date")).alias("Day"),
    F.date_format(F.col("Date"), "E").cast("string").alias("DayName"),
    F.quarter(F.col("Date")).alias("Quarter"),
    F.col("FiscalPeriodName").alias("FiscalPeriodName"),
    F.col("FiscalStartDate"),
    F.col("FiscalEndDate"),
    F.col("FiscalMonth"),
    F.col("FiscalYearStart"),
    F.col("FiscalYearEnd"),
    F.col("FiscalQuarter"),
    F.col("FiscalQuarterStart"),
    F.col("FiscalQuarterEnd"),
    F.concat(
    F.lit("FY"),
    F.year(F.col("FiscalYearEnd")).cast("string")
    ).alias("FiscalYear"),
    F.lit(UpdatedDateTime).alias("UpdatedDateTime"),
    F.xxhash64("DateId").alias("DateKey")
)

display(datedimdf)

###Final dataframe

In [0]:
df_final = datedimdf

## Write to Silver Schema

In [0]:
saveDeltaTableToCatalog(df_final,"silver",Entity)

In [0]:
%sql
SELECT FiscalYear, COUNT(*) AS row_count
FROM silver.dimdate
WHERE FiscalYear IS NOT NULL
GROUP BY FiscalYear
ORDER BY FiscalYear;